## The performance of a lock complex
In this notebook, we simulate a lock object on a network. Randomly generated, evenly sized vessels sampled from an exponential distribution have to pass this lock. We add a complex lock object to the graph. Vessels are levelled in the same lock operation if they can fit inside the lock, and register with the lock operator before the start of the lock operation. Based on this behaviour, we will quantify the performance of the lock in terms of efficiency and capacity.

In [1]:
# package(s) used for creating and geo-locating the graph
import networkx as nx
import pyproj
from shapely.geometry import Point, LineString, Polygon
from shapely.ops import transform

# package(s) related to the simulation (creating the vessel, running the simulation)
import datetime
import simpy
import opentnsim
from opentnsim.core.utils import create_object
from opentnsim.utils import inspect_object, generate_vessels_from_distribution
from opentnsim.graph import mixins as graph_module
from opentnsim.core.logutils import logbook2eventtable
from opentnsim.core import Identifiable, Movable, VesselProperties, ExtraMetadata
from opentnsim.core.visualizations import generate_vessel_gantt_chart
from opentnsim.graph.mixins import HasMultiDiGraph
from opentnsim.output import HasOutput
from scipy.stats import norm, uniform, expon

# import of modules important for locking
from opentnsim.lock import IsLockChamber, IsLockWaitingArea, IsLockComplex, LockComplexTraversable
from opentnsim.lock.calculations import estimate_lock_capacity, calculate_lock_occupancy
from opentnsim.lock.logutils import calculate_cycle_information, get_vessel_delays
from opentnsim.lock.visualizations import show_results

# package(s) needed for inspecting the output
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("This notebook is executed with OpenTNSim version {}".format(opentnsim.__version__))

This notebook is executed with OpenTNSim version 1.3.4


#### 0. Create environment

In [2]:
# start simpy environment
simulation_start = datetime.datetime(2025, 1, 1, 0, 0, 0)
env = simpy.Environment(initial_time=simulation_start.timestamp())
env.epoch = simulation_start

#### 1. Create graph
We create a directional graph with a layout similar to that of notebooks 0201, 0202, and 0203. However, we add two additional edges prior to nodes 0 and 1 to show that a lock can also be implemented in a larger graph. These edges are 20km long.

In [3]:
# define reference systems
wgs84eqd = pyproj.CRS('4087')
wgs84rad = pyproj.CRS('4326')

# define transformer functions
wgs84eqd_to_wgs84rad = pyproj.transformer.Transformer.from_crs(wgs84eqd,wgs84rad,always_xy=True).transform #equidistant wgs84 to radial wgs84
wgs84rad_to_wgs84eqd = pyproj.transformer.Transformer.from_crs(wgs84rad,wgs84eqd,always_xy=True).transform #radial wgs84 to equidistant wgs84

# create a directed graph
graph = nx.DiGraph()

# add nodes
graph.add_node('-1',geometry=transform(wgs84eqd_to_wgs84rad,Point(-25000,0)))
graph.add_node('0',geometry=transform(wgs84eqd_to_wgs84rad,Point(-5000,0)))
graph.add_node('1',geometry=transform(wgs84eqd_to_wgs84rad,Point(5000,0)))
graph.add_node('+1',geometry=transform(wgs84eqd_to_wgs84rad,Point(25000,0)))

# add edges
graph.add_edge('-1','0', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-25000, 0),Point(-5000, 0)])), weight=1, length_m=25000-5000)
graph.add_edge('0','-1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-5000, 0),Point(-25000, 0)])), weight=1, length_m=25000-5000)
graph.add_edge('0','1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(-5000, 0),Point(5000, 0)])), weight=1, length_m=10000)
graph.add_edge('1','0', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(5000, 0),Point(-5000, 0)])), weight=1, length_m=10000)
graph.add_edge('1','+1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(5000, 0),Point(25000, 0)])), weight=1, length_m=25000-5000)
graph.add_edge('+1','1', geometry = transform(wgs84eqd_to_wgs84rad, LineString([Point(25000, 0),Point(5000, 0)])), weight=1, length_m=25000-5000)

# add graph to environment
env.graph = graph

In [4]:
graph_module.plot_graph(graph)

#### 1+ Adding infrastructure
We construct the same lock complex as before.

In [5]:
lock_chamber = IsLockChamber(env=env,
                             lock_depth = 10,
                             name='Lock',
                             gate_open = '0',
                             edge = ('0','1'),
                             geometry_m = Polygon([Point(-200, -25),Point(-200, 25),Point(200, 25),Point(200, -25)]))

In [6]:
# The minimum required input for a lock complex are waiting areas at both sides of the lock
waiting_area_A = IsLockWaitingArea(env=env,
                                   name = 'Waiting area A',
                                   edge = ('0','1'),
                                   distance_from_edge_start = 0)

waiting_area_B = IsLockWaitingArea(env=env,
                                   name = 'Waiting area B',
                                   edge = ('1','0'),
                                   distance_from_edge_start = 0)

In [7]:
lock_complex = IsLockComplex(lock_chambers = [lock_chamber],
                             waiting_areas = [waiting_area_A, waiting_area_B],
                             registration_nodes = ['0','1'],
                             env=env,
                             name = 'Lock complex',)

#### 2. Create agents
We create the same vessel agent object.

In [8]:
# make your preferred Vessel class out of available mix-ins.
Vessel = create_object(
    "Vessel", 
    (
        LockComplexTraversable,     # allows to interact with a lock
        Identifiable,               # allows to give the object a name and a random ID,
        Movable,                    # allows the object to move, with a fixed speed, while logging this activity
        VesselProperties,           # allows vessel to have dimensions, namely a length (L), width (B), and draught (T)
        ExtraMetadata,              # allow additional information, such as an arrival time (required for passing a lock)
        HasMultiDiGraph,            # allow to operate on a graph that can include parallel edges from and to the same nodes
        HasOutput,                  # allow additional output to be stored
    ), 
)

In [9]:
def mission(env, vessel):
    """
    Method that defines the mission of the vessel.
    
    In this case: 
        keep moving along the path until its end point is reached
    """
    while True:
        yield from vessel.move()
        
        if vessel.geometry == nx.get_node_attributes(env.graph, "geometry")[vessel.route[-1]]:
            break

To better illustrate the lock's performance, we add two vessel generators at the network boundaries. From each direction, 10 equally sized vessels are generated randomly from an exponential distribution with a mean arrival rate of 30 minutes. We use different seeds to get different arrival times.

In [10]:
upstream_vessels = generate_vessels_from_distribution(env=env,
                                                      VesselClass = Vessel,
                                                      vessel_parameters = {'v':4, 'L':100, 'B':20, 'T':5, 'type':'tanker'},
                                                      mean_arrival_rate=30.,
                                                      number_of_vessels=10,
                                                      start_node = '-1',
                                                      end_node = '+1',
                                                      seed = 123)

downstream_vessels = generate_vessels_from_distribution(env=env,
                                                        VesselClass = Vessel,
                                                        vessel_parameters = {'v':4, 'L':100, 'B':20, 'T':5, 'type':'tanker'},
                                                        mean_arrival_rate=30.,
                                                        number_of_vessels=10,
                                                        start_node = '+1',
                                                        end_node = '-1',
                                                        seed = 456)

vessels = upstream_vessels + downstream_vessels

for vessel in vessels:
    env.process(mission(env, vessel))

#### 3. Run simulation
We run the simulation.

In [11]:
env.run()

#### 4. Inspect output
We skip the logbooks and inspect the Gantt charts and time-distance diagram.

##### Gantt chart of event table
The Gantt chart becomes quite full

In [12]:
df_eventtable = opentnsim.core.logutils.logbook2eventtable([*vessels, lock_chamber])
fig = generate_vessel_gantt_chart(df_eventtable)

##### Time-distance diagram of vessels passing the lock and planning info
Luckily, the time-distance diagram shows the behaviour of the vessels in a more comprehensible manner. Waiting times emerge.

In [13]:
def cm_to_pixels(cm):
    return cm * 37.8 # Set figure height to 10 cmfig.update_layout(height=cm_to_pixels(10))

# We can plot the time-distance diagram
fig = lock_chamber.plot(xlimmin = -6050, 
                        xlimmax = 6050,
                        ylimmin = pd.Timestamp('2025-01-01 00:00:00'),
                        ylimmax = pd.Timestamp('2025-01-01 10:00:00'),
                        method='Plotly',
                        boundary_nodes = ['-1','+1'])

fig.update_layout(height=cm_to_pixels(20))

##### Lock performance
We can simply get the performance using the <i>get_performance</i> function of the <strong>lock chamber</strong> object. This function calculates various KPI (key performance indicators):
- <strong>Vessel delay</strong>:
we calculate observed delays of vessels relative to their normal cruising speed at the edge. Delays emerge due to waiting times in the waiting areas and at the lock, levelling times, and sailing in and out of the lock at reduced speeds.
- <strong>Lock occupancy</strong>:
we calculate the lock occupancy by dividing the sum of the lengths of the vessels by the length dimension of the lock chamber.
- <strong>I/C ratio</strong> (traffic load, see Section 3.1 of Lecture Notes):
we calculate the intensity and divide it by the estimated lock capacity, both expressed as the number of vessels that pass the lock per hour. The intensity is calculated by identifying the lock operation cycles, their durations and the number of levelled vessels. The capacity is calculated based on the operation times, including loop times, sailing in times and gaps between vessels, and levelling times (see example box 3.1 of the Lecture Notes).

We start by calculating information for each cycle. We determine each cycle, including how long each lock process took (i.e., start and stop times, directions, loop times, sailing-in times, closing doors, levelling, opening doors, sailing-out times, and how many vessels were levelled).

In [14]:
Tc_df = calculate_cycle_information(lock_chamber)
Tc_df

,Start time of cycle,Stop time of cycle,Direction first operation,Direction second operation,Loop time start side,Sailing-in time start side,Closing gate time start side,Levelling time to opposing side,Opening gate time opposing side,Sailing-out time opposing side,...,Closing gate time opposing side,Levelling time to start side,Opening gate time start side,Sailing-out time start side,Cycle duration,Number of upstream vessels,Number of downstream vessels,Upstream vessel_ids,Downstream vessel_ids,Intensity (I_s)
0,2025-01-01 01:39:36.854414000,2025-01-01 03:03:12.602119,0,1,0 days 00:00:00,0 days 00:00:00,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:00:00,...,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:09:49,0 days 01:23:36,0,4,[],"[b1d400d8-5853-4aaa-a578-dbbb6a1bd8d3, 1d6232d...",2.870958
1,2025-01-01 01:59:36.854414000,2025-01-01 03:45:54.794343,1,0,0 days 00:00:00,0 days 00:33:47,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:09:49,...,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:09:49,0 days 01:46:18,4,4,"[b1d400d8-5853-4aaa-a578-dbbb6a1bd8d3, 1d6232d...","[ca34208d-3c25-4b63-8800-46caf8115145, ef22471...",4.515565
2,2025-01-01 03:03:12.602119000,2025-01-01 04:28:10.358619,0,1,0 days 00:03:05,0 days 00:09:49,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:09:49,...,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:06:49,0 days 01:24:58,4,3,"[ca34208d-3c25-4b63-8800-46caf8115145, ef22471...","[cbcc766f-76e6-4c3c-85ea-7b1b608fc1fc, 98b878b...",4.943351
3,2025-01-01 03:45:54.794344000,2025-01-01 05:02:06.935293,1,0,0 days 00:04:33,0 days 00:10:54,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:06:49,...,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:03:49,0 days 01:16:12,2,3,"[cbcc766f-76e6-4c3c-85ea-7b1b608fc1fc, 98b878b...","[27772acc-8a3f-4975-b755-d3c6963bbba4, 3a811ae...",3.936887
4,2025-01-01 04:28:10.358620000,2025-01-01 06:00:19.935553,0,1,0 days 00:03:05,0 days 00:07:03,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:03:49,...,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:00:49,0 days 01:32:10,2,1,"[27772acc-8a3f-4975-b755-d3c6963bbba4, 3a811ae...",[94b3e74d-9440-4fd3-98d0-a64821d116e6],1.953133
5,2025-01-01 05:02:06.935293001,2025-01-01 06:58:28.919978,1,0,0 days 00:31:44,0 days 00:05:40,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:00:49,...,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:09:49,0 days 01:56:22,4,1,[94b3e74d-9440-4fd3-98d0-a64821d116e6],"[32abbad2-2d06-46f2-bcee-c1e396728b0f, 3ce8e81...",2.578064
6,2025-01-01 06:00:19.935553000,2025-01-01 07:32:25.496651,0,1,0 days 00:03:05,0 days 00:25:15,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:09:49,...,0 days 00:05:00,0 days 00:10:00,0 days 00:05:00,0 days 00:03:49,0 days 01:32:06,4,2,"[32abbad2-2d06-46f2-bcee-c1e396728b0f, 3ce8e81...","[47f324bd-520c-4beb-b194-94385e1baa3c, 072729b...",3.909105


This is the information for the second cycle:

In [15]:
Tc_df.loc[1]

Start time of cycle                                       2025-01-01 01:59:36.854414
Stop time of cycle                                        2025-01-01 03:45:54.794343
Direction first operation                                                          1
Direction second operation                                                         0
Loop time start side                                                 0 days 00:00:00
Sailing-in time start side                                           0 days 00:33:47
Closing gate time start side                                         0 days 00:05:00
Levelling time to opposing side                                      0 days 00:10:00
Opening gate time opposing side                                      0 days 00:05:00
Sailing-out time opposing side                                       0 days 00:09:49
Loop time opposing side                                              0 days 00:03:05
Sailing-in time opposing side                                    

We can also get the aggregated cycle information

In [16]:
lock_chamber.get_aggregated_cycle_information()

Number of cycles                                                   3
Number of vessels                                                 18
Minimum cycle time                                   0 days 01:16:12
Average cycle time                                   0 days 01:33:06
Maximum cycle time                                   0 days 01:56:22
Total cycle time                                     0 days 10:51:42
Average duration loop time upstream                  0 days 00:02:19
Average fraction loop time upstream (%)                         2.84
Average duration sailing in time upstream            0 days 00:10:32
Average fraction sailing in time upstream (%)                  12.93
Average duration gate closing time upstream          0 days 00:05:00
Average fraction gate closing time upstream (%)                 5.37
Average duration levelling time to downstream        0 days 00:10:00
Average fraction levelling time to downstream (%)              10.74
Average duration gate opening time

###### <strong>Vessel delay</strong>
We present the results of the vessel delay calculations in the following forms: all the components that contributed to the delay, and aggregated per lock complex area and cause. Delays are defined as such that the vessel speed deviates from its normal cruising speed.

In [17]:
vessel_delays, vessel_delay_locations, vessel_delays_causes = get_vessel_delays(lock_chamber)
vessel_delays

,operation_nr,vessel_id,total_delay,waiting time in waiting_area for other vessels (%),waiting time in waiting_area for available operation (%),delay due to sailing to lock gate (%),delay due to sailing to position in lock (%),waiting time for other vessels to sail into lock (%),waiting time for closing doors (%),waiting time for levelling (%),waiting time for opening doors (%),waiting time for other vessels to sail out of lock (%),delay due to sailing out of lock (%),delay due to sailing away from lock (%)
0,1,b1d400d8-5853-4aaa-a578-dbbb6a1bd8d3,0 days 00:54:28,0.0,2.83,0.0,7.73,51.62,9.18,18.36,9.18,0.00,1.10,0.0
1,1,1d6232d2-38ef-4247-b851-3a566da8c998,0 days 00:44:51,0.0,0.00,0.0,6.71,41.61,11.15,22.29,11.15,3.08,4.02,0.0
2,1,e8f86303-ef49-474a-8ec6-652ac911a305,0 days 00:40:46,0.0,0.00,0.0,4.43,32.35,12.27,24.53,12.27,6.77,7.38,0.0
3,1,ee121ce1-20ff-41b5-9ba4-e1525ea54005,0 days 00:28:57,0.0,0.00,0.0,2.08,0.00,17.27,34.54,17.27,14.30,14.54,0.0
4,2,ca34208d-3c25-4b63-8800-46caf8115145,0 days 01:34:00,0.0,69.20,0.0,4.48,4.40,5.32,10.64,5.32,0.00,0.64,0.0
5,2,ef224713-59cd-4dad-8b8e-673bb8019b99,0 days 01:33:30,0.0,69.03,0.0,3.22,2.95,5.35,10.70,5.35,1.48,1.93,0.0
6,2,3fdd4c8d-7589-4a5a-96bc-be731f0be0a0,0 days 01:28:56,0.0,67.45,0.0,2.03,1.55,5.62,11.24,5.62,3.10,3.38,0.0
7,2,bdfad98e-857b-40c8-8e4e-76748100edd5,0 days 01:22:42,0.0,64.99,0.0,0.73,0.00,6.05,12.09,6.05,5.01,5.09,0.0
8,3,cbcc766f-76e6-4c3c-85ea-7b1b608fc1fc,0 days 00:30:03,0.0,0.00,0.0,14.01,17.43,16.64,33.28,16.64,0.00,2.00,0.0
9,3,98b878b7-d02e-469d-bbcf-b41c4eae33da,0 days 00:31:34,0.0,4.81,0.0,9.53,12.22,15.84,31.68,15.84,4.37,5.72,0.0


We observe that most delay takes place in the waiting area.

In [18]:
vessel_delay_locations

nr_operations                          7
nr_vessels                            20
min_vessel_delay         0 days 00:24:49
average_vessel_delay     0 days 01:02:21
max_vessel_delay         0 days 02:33:09
total_delay              0 days 20:47:07
waiting_area (%)                   46.86
sailing_to_lock (%)                 4.54
in_lock (%)                        45.42
sailing_from_lock (%)               3.18
dtype: object

This is because most waiting time is related to congestion. In addition, the lock operation takes significant time and is augmented by additional waiting time for other vessels to sail into/out of the lock chamber. The reduced sailing speed for manoeuvring into/out of the lock contributes to this.

In [19]:
vessel_delays_causes

nr_operations                          7
nr_vessels                            20
min_vessel_delay         0 days 00:24:49
average_vessel_delay     0 days 01:02:21
max_vessel_delay         0 days 02:33:09
total_delay              0 days 20:47:07
congestion (%)                     46.86
obstruction (%)                     7.72
traffic (%)                        13.35
operation of lock (%)              32.07
dtype: object

###### <strong>Lock occupancy</strong>
We show the results of the calculation of the occupancy of the lock (for each operation, and the average value):

In [20]:
occupancy, occupancy_df = calculate_lock_occupancy(lock_chamber)
display(occupancy_df)
print(f"The mean occupancy equals {occupancy} %")

,leveling_start,leveling_stop,Number of vessels,Lock length,Lock length claimed by vessels,Lock occupancy
0,2025-01-01 01:43:04.354414,2025-01-01 01:53:04.354414,0,400,0.0,0.00
1,2025-01-01 02:38:24.006006,2025-01-01 02:48:24.006006,4,400,400.0,1.00
2,2025-01-01 03:21:06.198231,2025-01-01 03:31:06.198231,4,400,400.0,1.00
3,2025-01-01 04:06:21.762507,2025-01-01 04:16:21.762507,3,400,300.0,0.75
4,2025-01-01 04:43:18.339181,2025-01-01 04:53:18.339181,2,400,200.0,0.50
5,2025-01-01 05:44:31.339441,2025-01-01 05:54:31.339441,1,400,100.0,0.25
6,2025-01-01 06:33:40.323865,2025-01-01 06:43:40.323865,4,400,400.0,1.00
7,2025-01-01 07:13:36.900539,2025-01-01 07:23:36.900539,2,400,200.0,0.50


The mean occupancy equals 62.5 %


###### <strong>I/C ratio</strong>
We show the results of the calculation of the capacity of the lock:

In [21]:
capacity, cycle_duration, cycle_duration_in_detail = estimate_lock_capacity(lock_chamber)
print(f"The capacity of the lock is {np.round(capacity,2)} vessels/hour \n")
print(f"The cycle duration equals {cycle_duration['Cycle duration']}, caused by:")
display(pd.Series(cycle_duration).drop("Cycle duration"))
print()
print("The following information was used/estimated to determine the capacity:")
display(cycle_duration_in_detail)

The capacity of the lock is 5.62 vessels/hour 

The cycle duration equals 0 days 01:25:24, caused by:


Entering time (%)     26.58
Operation time (%)    46.83
Exiting time (%)      26.58
dtype: object


The following information was used/estimated to determine the capacity:


Number of vessels in lock                                                            4.0
Sailing distance from crossing point to first gate (first vessel)                  370.0
Sailing speed from crossing point to first gate (first vessel)                         4
Sailing time from crossing point to first gate (first vessel)            0 days 00:01:32
Time gap between vessels sailing in                                      0 days 00:03:00
Total time of vessels sailing in from first vessel until last vessel     0 days 00:09:00
Distance from first lock gate to position in lock (last vessel)                     50.0
Sailing-in speed in lock (last vessel)                                          1.028889
Sailing time from first lock gate to position in lock (last vessel)      0 days 00:00:49
Closing gate time                                                                  300.0
Levelling time                                                                     600.0
Opening gate time    

We can also determine the intensity <i>I_s</i> for each half lock operation cycle

In [22]:
display(Tc_df[['Start time of cycle', 'Stop time of cycle', 'Direction first operation', 'Direction second operation', 
               'Number of upstream vessels', 'Number of downstream vessels', 'Intensity (I_s)']])
print(f"This results in an cycle-average intensity of: {np.round(Tc_df['Intensity (I_s)'].mean(),2)} vessels/hour")

,Start time of cycle,Stop time of cycle,Direction first operation,Direction second operation,Number of upstream vessels,Number of downstream vessels,Intensity (I_s)
0,2025-01-01 01:39:36.854414000,2025-01-01 03:03:12.602119,0,1,0,4,2.870958
1,2025-01-01 01:59:36.854414000,2025-01-01 03:45:54.794343,1,0,4,4,4.515565
2,2025-01-01 03:03:12.602119000,2025-01-01 04:28:10.358619,0,1,4,3,4.943351
3,2025-01-01 03:45:54.794344000,2025-01-01 05:02:06.935293,1,0,2,3,3.936887
4,2025-01-01 04:28:10.358620000,2025-01-01 06:00:19.935553,0,1,2,1,1.953133
5,2025-01-01 05:02:06.935293001,2025-01-01 06:58:28.919978,1,0,4,1,2.578064
6,2025-01-01 06:00:19.935553000,2025-01-01 07:32:25.496651,0,1,4,2,3.909105


This results in an cycle-average intensity of: 3.53 vessels/hour


###### <strong>Aggregated lock performance</strong>
The observed KPIs show that there is some significant delay, as the I/C-ratio is high (0.63). The cycle times are significant, particularly due to the duration of sailing in of vessels (29.46%), gate movements (21.48%) and levelling (21.48%).

In [23]:
summary = lock_chamber.get_performance()